In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Install the dependencies


### Grasp point generation dependencies

In [3]:
!nvidia-smi
%pip install ultralytics
import ultralytics
ultralytics.checks()
import os
HOME = os.getcwd()
print("HOME:", HOME)
%pip install -q 'git+https://github.com/facebookresearch/segment-anything.git'
%pip install -q jupyter_bbox_widget roboflow dataclasses-json supervision
!mkdir -p {HOME}/weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -P {HOME}/weights
CHECKPOINT_PATH = os.path.join(HOME, "weights", "sam_vit_h_4b8939.pth")
print(CHECKPOINT_PATH, "; exist:", os.path.isfile(CHECKPOINT_PATH))
import torch
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
MODEL_TYPE = "vit_h"

Ultralytics YOLOv8.2.50 🚀 Python-3.10.12 torch-2.3.0+cu121 CUDA:0 (Tesla T4, 15102MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 30.2/78.2 GB disk)
HOME: /content
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.8/367.8 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.2/76.2 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.0/124.0 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.1 MB/s eta 0:00:00
/content/weights/sam_vit_h_4b8939.pth ; exist: True


### Image captioning and Instruction Generation dependencies


In [4]:
!pip install scikit-learn nltk
!pip install openai
!pip install -q git+https://github.com/huggingface/peft.git transformers bitsandbytes datasets


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 8.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 46.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.1/314.1 kB 33.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 14.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1

## Grasp Point Genration

In [5]:
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor
from ultralytics import YOLO
import numpy as np
import cv2 as cv
import random
from math import atan2, cos, sin, sqrt, pi , tan , radians
from shapely.geometry import LineString,Polygon
from google.colab.patches import cv2_imshow
from itertools import combinations
import supervision as sv
from PIL import Image

In [6]:
import time
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(device=DEVICE)
mask_generator = SamAutomaticMaskGenerator(sam)
mask_predictor = SamPredictor(sam)
desired_label=["remote","bottle","cup","bowl","vase","scissors","toothbrush","wine glass"]
def process_image(IMAGE_PATH,desired_label):
  model = YOLO("yolov8n.pt")
  results = model([IMAGE_PATH])
  for result in results:
    boxes = result.boxes
    if(boxes.shape[0]==0):
      return [0,0,0,0]
    for i in range(boxes.shape[0]):
        if (model.names[int(boxes.cls[i].item())] in desired_label):
          bounding_box=np.array([(boxes.xyxy[i].cpu().numpy()[0]),
                        (boxes.xyxy[i].cpu().numpy()[1]),
                        (boxes.xyxy[i].cpu().numpy()[2]),
                        (boxes.xyxy[i].cpu().numpy()[3])])

          return bounding_box
        else:
          return [0,0,0,0]

def sam_segment(bbox,IMAGE_PATH):
  image_bgr = cv.imread(IMAGE_PATH)
  image_rgb = cv.cvtColor(image_bgr, cv.COLOR_BGR2RGB)
  mask_predictor.set_image(image_rgb)
  masks, scores, logits = mask_predictor.predict(
      box=bbox,
      multimask_output=False
  )
  detections = sv.Detections(
    xyxy=sv.mask_to_xyxy(masks=masks),
    mask=masks
  )
  detections = detections[detections.area == np.max(detections.area)]
  return detections
def calculate_angle_and_coordinates(ann,image,bbox):
      images=[]
      bw=ann.mask[0].astype(int)
      bw = np.array(bw, np.uint8)
      contours, _ = cv.findContours(bw, cv.RETR_LIST, cv.CHAIN_APPROX_NONE)
      points=[]
      max_area=0
      for k, c in enumerate(contours):
        area = cv.contourArea(c)
        if max_area<area:
          rect = cv.minAreaRect(c)
          box = cv.boxPoints(rect)
          box = np.int0(box)
          width = int(rect[1][0])
          height = int(rect[1][1])
          angle = int(rect[2])
          if np.sqrt(pow(box[0][0]-box[0][1],2)+pow(box[0][1]-box[1][1],2))<np.sqrt(pow(box[2][0]-box[3][0],2)+pow(box[2][1]-box[3][1],2)):
              angle = 90+angle
          else:
              angle=-angle
          x=random.randint(int(bbox[0]),int(bbox[2]))
          y=random.randint(int(bbox[1]),int(bbox[3]))
          x_limit=list(range(x-1024,x+1024))
          x_limit=np.array(x_limit)
          y_limit=[]
          if (angle != 90 or angle!=-90):
              for k in range(-1024,1024):
                  y_limit.append(y-tan( radians(angle))*k)
          else:
              for k in range(-1024,1024):
                  y_limit.append(y)
          max_area=area
      for k, c in enumerate(contours):
        c=np.reshape(c,(-1,2))
        line_1=LineString(np.column_stack((x_limit,y_limit)))
        if(len(c)<4):
          return None
        line_2=Polygon(c)
        intersection=line_1.intersection(line_2,grid_size=1)
        if type(intersection) is LineString:
          if intersection:
            points.append([intersection.xy[0][0],intersection.xy[1][0]])
            points.append([intersection.xy[0][-1],intersection.xy[1][-1]])
      points.sort()
      if points:
        image_copy=image.copy()
        cv.circle(image_copy,(int(points[0][0]),int(points[0][1])),25,(0,255,0),-1)
        cv.circle(image_copy,(int(points[1][0]),int(points[1][1])),25,(0,255,0),-1)
        images.append(image_copy)
      if (len(points)>2):
        image_copy=image.copy()
        cv.circle(image_copy,(int(points[-1][0]),int(points[-1][1])),25,(0,255,0),-1)
        cv.circle(image_copy,(int(points[-2][0]),int(points[-2][1])),25,(0,255,0),-1)
        images.append(image_copy)
      return images

def grasp_point_gen(image_path,output_folder):
  first=time.time()
  count=0
  image_bgr = cv.imread(image_path)
  image_rgb = cv.cvtColor(image_bgr, cv.COLOR_BGR2RGB)
  bbox=process_image(image_path,desired_label)
  if(type(bbox)!=list):
    detections=sam_segment(bbox,image_path)
    while(True):
      if count<30:
        if(time.time()-first<90):
          image = calculate_angle_and_coordinates(detections,image_rgb,bbox)
          if(image is not None):
            for j in range(len(image)):
              name = f'{count}.jpg'
              output_path = os.path.join(output_folder, name)
              cv.imwrite(output_path, image[j])
              count=count+1
        else:
          name = f'{0}.jpg'
          output_path = os.path.join(output_folder, name)
          cv.imwrite(output_path, image_rgb)
          break
      else:
        break
  else:
      name = f'{0}.jpg'
      output_path = os.path.join(output_folder, name)
      cv.imwrite(output_path, image_rgb)

In [7]:
def generate_images(output_path,image_path):
  grasp_point_gen(image_path,output_path)
  grasp_points=[]
  for filename in os.listdir(output_path):
      image_path = os.path.join(output_path, filename)
      grasp_points.append(Image.open(image_path))
  return grasp_points

In [8]:
def calculate_angle_and_coordinates_geo(ann,image,bbox):
      images=[]
      bw=ann.mask[0].astype(int)
      bw = np.array(bw, np.uint8)
      contours, _ = cv.findContours(bw, cv.RETR_LIST, cv.CHAIN_APPROX_NONE)
      points=[]
      max_area=0
      for k, c in enumerate(contours):
        area = cv.contourArea(c)
        rect = cv.minAreaRect(c)
        box = cv.boxPoints(rect)
        box = np.int0(box)
        center = (int(rect[0][0]),int(rect[0][1]))
        width = int(rect[1][0])
        height = int(rect[1][1])
        angle = int(rect[2])
        actual_angle=angle
        if width > height:
            angle = 90+angle
        else:
            angle=-angle
        x=int(rect[0][0])
        y=int(rect[0][1])
        x_limit=list(range(x-400,x+400))
        x_limit=np.array(x_limit)
        y_limit=[]
        if (angle != 90 or angle!=-90):
            for k in range(-400,400):
                y_limit.append(y-tan( radians(angle))*k)
        else:
            for k in range(-400,400):
                y_limit.append(y)
        line = []
        for x in range(0,np.size(x_limit)):
            row = []
            for x in range(2):
                row.append(0)
            line.append(row)
        c=np.reshape(c,(-1,2))
        line_1=LineString(np.column_stack((x_limit,y_limit)))
        if(len(c)<4):
          return None
        line_2=Polygon(c)
        intersection=line_1.intersection(line_2,grid_size=1)
        if type(intersection) is LineString:
          if intersection:
            points.append([intersection.xy[0][0],intersection.xy[1][0]])
            points.append([intersection.xy[0][-1],intersection.xy[1][-1]])
      points.sort()
      if points:
        image_copy=image.copy()
        cv.circle(image_copy,(int(points[0][0]),int(points[0][1])),25,(0,255,0),-1)
        cv.circle(image_copy,(int(points[-1][0]),int(points[-1][1])),25,(0,255,0),-1)
        images.append(image_copy)
      else:
        images.append(image)
      return images

In [9]:
def grasp_point_gen_geo(image_path,output_folder):
  image_bgr = cv.imread(image_path)
  image_rgb = cv.cvtColor(image_bgr, cv.COLOR_BGR2RGB)
  bbox=process_image(image_path,desired_label)
  if(type(bbox)!=list):
    detections=sam_segment(bbox,image_path)
    image = calculate_angle_and_coordinates_geo(detections,image_rgb,bbox)
    name = f'geo.jpg'
    output_path = os.path.join(output_folder, name)
    if(image is not None):
      cv.imwrite(output_path, image[0])
      return output_path
    else:
      cv.imwrite(output_path, image_rgb)
      return output_path
  else:
      name = f'geo.jpg'
      output_path = os.path.join(output_folder, name)
      cv.imwrite(output_path, image_rgb)
      return output_path

In [10]:
def generate_random(output_path,image_path):
  image_path_random = os.path.join(output_path, f"0.jpg")
  return image_path_random

## Task Orinted Grasp Point Generation and the Baselines Infereance and evaluation

### Task Orinted Grasp Point scripts

In [11]:
%run few_shot_instruction_generation.py
%run fine_tuned_blip_inference.py


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.60k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

In [12]:
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
scoring_model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

def get_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        outputs = scoring_model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.numpy()

"""## Find the best grasp score"""

def task_orinted_grasp_point(input_image_path,out_put_path,user_input):

  #get the instructions
  input_caption = generate_caption(Image.open(input_image_path),base_model)
  instructions = generate_instruction(user_input,input_caption)

  #extract the grasping location from the instructions

  parts = instructions.split("grasping location:")

  task_details = parts[0].strip()

  grasping_location = "grasping location a " + parts[1].strip()

  #get the grasp points

  grasp_points = generate_images(output_path,image_path)

  #find the best grasp point
  grasp_scores = []
  for image in grasp_points:
    caption = generate_caption(image,fine_tuned_model)
    # Get embeddings for the caption and the grasping location
    caption_embedding = get_embedding(caption)
    grasping_location_embedding = get_embedding(grasping_location)

    # Calculate cosine similarity
    cosine_sim = cosine_similarity(caption_embedding, grasping_location_embedding)
    grasp_scores.append([image,caption, cosine_sim[0][0]])


  highest_score_item = max(grasp_scores, key=lambda x: x[1])

  #return the best grasping location
  return highest_score_item


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

### Evaluation

In [13]:
def evaluation_caption(image_base64):
  system_message = {
      "role": "system",
      "content": (
          "determine where the grasping location is based on the locations of the green markers on the input image.\n\n"
          "###EXAMPLES###\n\n"
          "input image: image of a cup with green markers on the handle\n"
          "grasping location: cup's handle\n\n"
          "input image: image of a cup with green markers on the body\n"
          "grasping location: cup's body\n\n"
          "input image: image of a cup with green markers on the rim(edge)\n"
          "grasping location: cup's edge \n\n"
          "input image: image of a cup with green markers on the base of the cup\n"
          "grasping location:  cup's base\n\n"
          "input image: image of an object with no green markers\n"
          "grasping location: not found\n\n"
          "input image: image with green markers on the neck of a vase\n"
          "grasping location: vase's neck\n\ninput image: image with green markers on the neck of a bottle\n"
          "grasping location: bottle's neck\n"
      )
  }

  user_message = {
      "role": "user",
      "content": [
        {
          "type": "image_url",
          "image_url": {
              "url": f"data:image/jpeg;base64,{image_base64}"         }
        }
      ]
  }

  messages = [system_message, user_message]

  try:
    client = openai.OpenAI(
        api_key = OPENAI_API_KEY
    )
    response =  client.chat.completions.create(
          model="gpt-4o",
          messages=messages,
          temperature=1,
          max_tokens=256,
          top_p=1,
          frequency_penalty=0,
          presence_penalty=0
      )
    return response.choices[0].message.content

  except Exception as e:
    print(e)



In [14]:
import base64
import requests

def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

In [15]:
def calculate_cosine_sim(text1,text2):
  text1_embedding = get_embedding(text1)
  text2_embedding = get_embedding(text2)

  # Calculate cosine similarity
  cosine_sim = cosine_similarity(text1_embedding, text2_embedding)
  return cosine_sim[0][0]

In [ ]:
import csv

testset = '/content/drive/MyDrive/LLM final project/testset.csv'
output_path = "/content/output_dir"

task_orinted_cosines = []
rand_cosines = []
geo_cosines = []
with open(testset, mode='r', newline='') as file:
    reader = csv.reader(file)

    next(reader)

    for i,row in  enumerate(reader):
        image_path = f"/content/drive/MyDrive/LLM final project/testset/{row[0]}"
        user_input = row[1]
        true_grasp_point = row[2]

        task_orinted = task_orinted_grasp_point(image_path,output_path,user_input)
        task_orinted_text = task_orinted[1]
        task_orinted_cosines.append(calculate_cosine_sim(true_grasp_point,task_orinted_text))

        output_path_random = generate_random(output_path,image_path)
        base64_rand = encode_image(output_path_random)
        rand_text = evaluation_caption(base64_rand)
        rand_cosines.append(calculate_cosine_sim(true_grasp_point,rand_text))


        output_path_geo = grasp_point_gen_geo(image_path,output_path)
        base64_geo = encode_image(output_path_geo)
        geo_text = evaluation_caption(base64_geo)
        geo_cosines.append(calculate_cosine_sim(true_grasp_point,geo_text))


In [18]:
task_orinted_cosine_avg = sum(task_orinted_cosines)/len(task_orinted_cosines)
rand_cosine_avg = sum(rand_cosines)/len(rand_cosines)
geo_cosine_avg = sum(geo_cosines)/len(geo_cosines)

print("Average cosine similraity for task orinted grasp points: ",task_orinted_cosine_avg)
print("Average cosine similraity for random grasp points: ",rand_cosine_avg)
print("Average cosine similraity for Gep grasp points: ",geo_cosine_avg)

Average cosine similraity for task orinted grasp points:  0.6654995705296354
Average cosine similraity for random grasp points:  0.5205110750365548
Average cosine similraity for Gep grasp points:  0.5213366008204657
